Analysing the drift 
drift_patterns_over_time=[[1, 2],
                         [1, 2],
                         [2, 1]],

In [1]:
from log_utils.logging import read_logs
import numpy as np

### Get Pre-drift data from logs

In [2]:
# This code gets the min, max, avg performace stats 
#  - for each simulation timestep,
#  - in all clients,
#  - from the beginning of the simulation unitl the timestep where drift starts

def compute_pre_drift_stats_all_clients(client_log, drift_start):
    """
    Pre-drift stats: for each round t in [0, drift_start),
    compute min/max/avg across ALL clients.
    """
    pre_stats = {}
    for t in range(drift_start):
        losses = [loss for loss, _ in client_log[t]]
        accs   = [acc  for _, acc in client_log[t]]

        pre_stats[t] = {
            "loss_min": min(losses),
            "loss_max": max(losses),
            "loss_avg": sum(losses) / len(losses),
            "acc_min":  min(accs),
            "acc_max":  max(accs),
            "acc_avg":  sum(accs) / len(accs),
        }
    return pre_stats

### Setting up client clusters
##### Arranging the client cluster according to the drift patterns

In [3]:
# This is a helper function

def extend_clusters_for_final_segment(clusters_per_segment):
    """
    compute_cluster_segmented_stats creates an extra segment [last_drift, T),
    so we reuse the last clustering pattern for that final segment.
    """
    return clusters_per_segment + [clusters_per_segment[-1]]

In [5]:
def reorder_clusters_by_pattern(
    drift_clustered_client_indices,
    drift_pattern,
    keep_unmentioned="append"  # "append" | "drop"
):
    """
    Reorders clusters at each step according to drift_pattern.

    Supports:
      - subset orders like [1]
      - repeated indices like [1,1]
      - normal permutations like [2,1]

    keep_unmentioned:
      - "append": append clusters not referenced in 'order' at the end (in original order)
      - "drop": only keep clusters referenced in 'order'
    """
    if len(drift_clustered_client_indices) != len(drift_pattern):
        raise ValueError("Length mismatch: clustered indices and pattern must have same #steps.")

    reordered = []

    for step_clusters, order in zip(drift_clustered_client_indices, drift_pattern):
        k = len(step_clusters)

        # Validate each requested cluster index
        for idx in order:
            if not (1 <= idx <= k):
                raise ValueError(f"Invalid cluster index {idx} for step with {k} clusters (valid: 1..{k}).")

        # Reorder (1-based -> 0-based). Allows duplicates.
        reordered_step = [step_clusters[i - 1] for i in order]

        if keep_unmentioned == "append":
            mentioned = set(order)
            # append unmentioned clusters in original order
            for i in range(1, k + 1):
                if i not in mentioned:
                    reordered_step.append(step_clusters[i - 1])
        elif keep_unmentioned == "drop":
            pass
        else:
            raise ValueError("keep_unmentioned must be 'append' or 'drop'.")

        reordered.append(reordered_step)

    return reordered


### Cluster (grouped-clients) stats after drift starts
##### Per cluster seperated stats 

In [10]:
def per_round_stats_for_clients_in_range(client_log, clients, start_t, end_t):
    """
    client_log[t][client] = (loss, acc)
    Computes per-round min/max/avg over the given 'clients' for t in [start_t, end_t).
    """
    per_round_cluster_stats  = {}
    for t in range(start_t, end_t):
        losses = [client_log[t][c][0] for c in clients]
        accs   = [client_log[t][c][1] for c in clients]

        per_round_cluster_stats[t] = {
            "loss_min": min(losses),
            "loss_max": max(losses),
            "loss_avg": sum(losses) / len(losses),
            "acc_min":  min(accs),
            "acc_max":  max(accs),
            "acc_avg":  sum(accs) / len(accs),
        }
    return per_round_cluster_stats 

In [11]:
def compute_cluster_segmented_stats(client_log, drift_steps, clusters_per_segment):
    """
    Compute 
        - cluster-wise 
        - per-round statistics, for,
        - each drift segment, 
        - INCLUDING the final segment from the last drift step to the end of the simulation.

    Inputs
    ------
    client_log:
        Nested list where client_log[t][client_id] = (loss, accuracy)
    drift_steps:
        List of drift timestamps, e.g., [20, 33, 35, 45]
        Segments are built as: [20,33), [33,35), [35,45), [45,T)
    clusters_per_segment:
        List aligned with segments. Each element contains clusters for that segment:
          clusters_per_segment[segment_idx][cluster_idx] = [client_ids...]

        IMPORTANT: Must include cluster definitions for the final [last_drift, T) segment as well.

    Output
    ------
    cluster_stats:
        cluster_stats[cluster_idx][(start, end)][t] = {
            "loss_min", "loss_max", "loss_avg",
            "acc_min",  "acc_max",  "acc_avg"
        }
    """
    T = len(client_log)

    # Ensure drift steps are in increasing order so segment construction is correct.
    drift_steps_sorted = sorted(drift_steps)

    if not drift_steps_sorted:
        raise ValueError("drift_steps is empty; cannot build drift segments without at least one drift step.")

    # Build segments between drift points PLUS the final segment to T:
    # e.g., [20,33), [33,35), [35,45), [45,T)
    segment_boundaries = drift_steps_sorted + [T]
    drift_segments = list(zip(segment_boundaries[:-1], segment_boundaries[1:]))

    # clusters_per_segment must provide cluster definitions for EACH segment (including the final one).
    if len(drift_segments) != len(clusters_per_segment):
        raise ValueError(
            f"Mismatch: {len(drift_segments)} drift segments but {len(clusters_per_segment)} cluster definitions."
        )

    # We assume the number of clusters is consistent across segments (e.g., always 3 after appending [8,9]).
    n_clusters = len(clusters_per_segment[0])
    for segment_idx, segment_clusters in enumerate(clusters_per_segment):
        if len(segment_clusters) != n_clusters:
            raise ValueError(
                "All segments must have the same number of clusters "
                "(e.g., after appending [8,9]). "
                f"Segment {segment_idx} has {len(segment_clusters)} clusters, expected {n_clusters}."
            )

    # Prepare output container:
    cluster_stats = {cluster_idx: {} for cluster_idx in range(n_clusters)}

    # For each segment, compute per-round stats for each cluster's client IDs.
    for (segment_start, segment_end), segment_clusters in zip(drift_segments, clusters_per_segment):
        for cluster_idx, client_ids in enumerate(segment_clusters):
            cluster_stats[cluster_idx][(segment_start, segment_end)] = per_round_stats_for_clients_in_range(
                client_log=client_log,
                clients=client_ids,
                start_t=segment_start,
                end_t=segment_end
            )

    return cluster_stats

In [12]:
# For easy plotting

def cluster_stats_to_plot_series(cluster_stats):
    """
    Converts:
      cluster_stats[k][(start,end)][t] = stats
    into:
      plot_data[k][(start,end)] = {"round": [...], "acc_avg":[...], "loss_avg":[...], ...}
    """
    plot_data = {}

    for k, seg_dict in cluster_stats.items():
        plot_data[k] = {}
        for (start, end), per_round in seg_dict.items():
            rounds = sorted(per_round.keys())
            plot_data[k][(start, end)] = {
                "round": rounds,
                "loss_min": [per_round[r]["loss_min"] for r in rounds],
                "loss_avg": [per_round[r]["loss_avg"] for r in rounds],
                "loss_max": [per_round[r]["loss_max"] for r in rounds],
                "acc_min":  [per_round[r]["acc_min"]  for r in rounds],
                "acc_avg":  [per_round[r]["acc_avg"]  for r in rounds],
                "acc_max":  [per_round[r]["acc_max"]  for r in rounds],
            }
    return plot_data

### Compile log data for performance plots 

In [14]:
def build_plot_inputs_for_log(client_log, drift_steps, clusters_per_segment_without_final):
    """
    Returns:
      pre_stats: dict[t] -> stats
      post_cluster_plot_stats: output of cluster_stats_to_plot_series(cluster_stats)
    """
    drift_steps_sorted = sorted(drift_steps)
    drift_start = drift_steps_sorted[0]

    pre_stats = compute_pre_drift_stats_all_clients(client_log, drift_start)

    clusters_per_segment = extend_clusters_for_final_segment(clusters_per_segment_without_final)

    cluster_stats = compute_cluster_segmented_stats(
        client_log=client_log,
        drift_steps=drift_steps_sorted,
        clusters_per_segment=clusters_per_segment
    )

    post_cluster_plot_stats = cluster_stats_to_plot_series(cluster_stats)
    return pre_stats, post_cluster_plot_stats

### Plot performance

Description: We have mainly 3 dimentional infomation to show.
   (1) loss and accuracy
   (2) algorithm
   (3) cluster ID or drift ID.
    
So we have the following plot techniques to represent them.
    (1) plot all info in 1 figure
    (2) separate figures for each cluster ID (drift type) but for all algorithms

1) Helper: convert stats dicts into plot arrays

In [15]:
import numpy as np
import matplotlib.pyplot as plt

def stats_dict_to_arrays(stats_by_round, metric_prefix):
    rounds = np.array(sorted(stats_by_round.keys()), dtype=int)
    y_min = np.array([stats_by_round[t][f"{metric_prefix}_min"] for t in rounds], dtype=float)
    y_avg = np.array([stats_by_round[t][f"{metric_prefix}_avg"] for t in rounds], dtype=float)
    y_max = np.array([stats_by_round[t][f"{metric_prefix}_max"] for t in rounds], dtype=float)
    return rounds, y_min, y_avg, y_max

2) Core plotting: one “series” (pre + post clusters) onto a given axis

In [16]:
def plot_pre_post_drift_continuous_on_axis(
    ax,
    *,
    pre_stats,
    post_cluster_plot_stats,
    metric_prefix,                  # "acc" or "loss"
    base_color,                     # one color per method
    pre_label,
    pre_linestyle="-",              # solid for pre-drift
    pre_linewidth=1.8,
    cluster_linestyles=None,        # dict: {cluster_idx: linestyle}
    cluster_labels=None,            # dict: {cluster_idx: label}
    cluster_linewidth=1.6,
    band_alpha=0.25,
    show_band=True,
    stitch=True,
    drift_steps=None,               # list of drift boundaries (vertical lines)
    collect_legend_handles=True,    # whether to collect legend handles for this method
    legend_mode="method",           # "method" | "method+clusters" | "none"
):
    """
    Adds pre-drift + post-drift cluster curves to an existing axis.
    Returns a list of legend handles according to legend_mode.
    """
    if cluster_linestyles is None:
        cluster_linestyles = {}
    if cluster_labels is None:
        cluster_labels = {}

    legend_handles = []

    # -------------------------
    # 1) Pre-drift (all clients)
    # -------------------------
    x_pre, y_pre_min, y_pre_avg, y_pre_max = stats_dict_to_arrays(pre_stats, metric_prefix)

    pre_line, = ax.plot(
        x_pre, y_pre_avg,
        color=base_color,
        linestyle=pre_linestyle,
        linewidth=pre_linewidth,
        label=pre_label
    )

    if collect_legend_handles and legend_mode in ("method", "method+clusters"):
        legend_handles.append(pre_line)

    if show_band:
        ax.fill_between(
            x_pre, y_pre_min, y_pre_max,
            color=base_color,
            alpha=band_alpha,
            linewidth=0
        )

    # Identify the last pre-drift round/value for stitching
    pre_last_round = int(x_pre[-1])
    pre_last_avg = float(y_pre_avg[-1])

    # -------------------------
    # 2) Find first post-drift round
    # -------------------------
    first_post_round = None
    for _, segments_dict in post_cluster_plot_stats.items():
        for (_, _), seg in segments_dict.items():
            if len(seg["round"]) > 0:
                candidate = int(seg["round"][0])
                first_post_round = candidate if first_post_round is None else min(first_post_round, candidate)

    # -------------------------
    # 3) Stitch pre -> post
    # -------------------------
    if stitch and first_post_round is not None and (pre_last_round + 1 == first_post_round):
        for _, segments_dict in post_cluster_plot_stats.items():
            cluster_first_avg = None
            for (_, _), seg in segments_dict.items():
                if len(seg["round"]) == 0:
                    continue
                if int(seg["round"][0]) == first_post_round:
                    cluster_first_avg = float(seg[f"{metric_prefix}_avg"][0])
                    break

            if cluster_first_avg is None:
                continue

            # Stitching line uses the SAME style as pre-drift
            ax.plot(
                [pre_last_round, first_post_round],
                [pre_last_avg, cluster_first_avg],
                color=base_color,
                linestyle=pre_linestyle,
                linewidth=pre_linewidth
            )

    # -------------------------
    # 4) Drift boundary markers
    # -------------------------
    if drift_steps is not None:
        for d in drift_steps:
            ax.axvline(
                x=d,
                color="black",
                linestyle=":",
                linewidth=0.8,
                alpha=0.6,
                zorder=0
            )

    # -------------------------
    # 5) Post-drift (cluster-wise), continuous across segments
    # -------------------------
    for cluster_idx, segments_dict in post_cluster_plot_stats.items():
        linestyle = cluster_linestyles.get(cluster_idx, "--")
        label = cluster_labels.get(cluster_idx, f"{pre_label} | Cluster {cluster_idx}")

        x_all, y_min_all, y_avg_all, y_max_all = [], [], [], []

        for (start, end) in sorted(segments_dict.keys()):
            seg = segments_dict[(start, end)]
            x_seg = np.array(seg["round"], dtype=int)

            y_min_seg = np.array(seg[f"{metric_prefix}_min"], dtype=float)
            y_avg_seg = np.array(seg[f"{metric_prefix}_avg"], dtype=float)
            y_max_seg = np.array(seg[f"{metric_prefix}_max"], dtype=float)

            x_all.append(x_seg)
            y_min_all.append(y_min_seg)
            y_avg_all.append(y_avg_seg)
            y_max_all.append(y_max_seg)

        if not x_all:
            continue

        x = np.concatenate(x_all)
        y_min = np.concatenate(y_min_all)
        y_avg = np.concatenate(y_avg_all)
        y_max = np.concatenate(y_max_all)

        cluster_line, = ax.plot(
            x, y_avg,
            color=base_color,
            linestyle=linestyle,
            linewidth=cluster_linewidth,
            label=label
        )

        # Only include cluster lines in legend when explicitly requested
        if collect_legend_handles and legend_mode == "method+clusters":
            legend_handles.append(cluster_line)

        if show_band:
            ax.fill_between(
                x, y_min, y_max,
                color=base_color,
                alpha=band_alpha,
                linewidth=0
            )

    return legend_handles

3. Plot

In [17]:
def plot_multiple_methods_pre_post_drift(
    *,
    method_configs,
    drift_steps,
    out_prefix,
    band_alpha=0.25,
    show_band=True,
    stitch=True,
):
    """
    Plots multiple methods (logs) on the same Accuracy and Loss figures and saves both.

    method_configs: list of dicts, each:
      {
        "name": "FedAvg",
        "pre_stats": <dict>,
        "post_cluster_plot_stats": <dict>,
        "color": "red",
        "cluster_linestyles": {0:"--",1:"-.",2:":"},
        "cluster_labels": {0:"...",1:"...",2:"..."}  (optional)
        "show_in_legend": True/False                (optional, default True)
        "legend_mode": "method" | "method+clusters" | "none"  (optional, default "method")
      }
    """
    drift_steps_sorted = sorted(drift_steps)

    fig_acc, ax_acc = plt.subplots()
    fig_loss, ax_loss = plt.subplots()

    acc_handles_all = []
    loss_handles_all = []

    for cfg_idx, cfg in enumerate(method_configs):
        pre_stats = cfg["pre_stats"]
        post_stats = cfg["post_cluster_plot_stats"]

        base_color = cfg.get("color", "black")
        name = cfg.get("name", "method")

        cluster_linestyles = cfg.get("cluster_linestyles", {})
        cluster_labels = cfg.get("cluster_labels", {})

        collect_legend = cfg.get("show_in_legend", True)
        legend_mode = cfg.get("legend_mode", "method")

        # Draw drift lines only once (on the first method) to avoid over-plotting
        drift_lines_for_this_call = drift_steps_sorted if cfg_idx == 0 else None

        acc_handles = plot_pre_post_drift_continuous_on_axis(
            ax_acc,
            pre_stats=pre_stats,
            post_cluster_plot_stats=post_stats,
            metric_prefix="acc",
            base_color=base_color,
            pre_label=f"{name} (pre-drift)",
            pre_linestyle="-",
            cluster_linestyles=cluster_linestyles,
            cluster_labels=cluster_labels,
            band_alpha=band_alpha,
            show_band=show_band,
            stitch=stitch,
            drift_steps=drift_lines_for_this_call,
            collect_legend_handles=collect_legend,
            legend_mode=legend_mode,
        )

        loss_handles = plot_pre_post_drift_continuous_on_axis(
            ax_loss,
            pre_stats=pre_stats,
            post_cluster_plot_stats=post_stats,
            metric_prefix="loss",
            base_color=base_color,
            pre_label=f"{name} (pre-drift)",
            pre_linestyle="-",
            pre_linewidth=1.8,
            cluster_linestyles=cluster_linestyles,
            cluster_labels=cluster_labels,
            band_alpha=band_alpha,
            show_band=show_band,
            stitch=stitch,
            drift_steps=drift_lines_for_this_call,
            collect_legend_handles=collect_legend,
            legend_mode=legend_mode,
        )

        # Legend collection (no duplicates)
        if collect_legend and acc_handles and loss_handles:
            if legend_mode == "method+clusters":
                acc_handles_all.extend(acc_handles)
                loss_handles_all.extend(loss_handles)
            elif legend_mode == "method":
                acc_handles_all.append(acc_handles[0])
                loss_handles_all.append(loss_handles[0])
            # legend_mode == "none" -> add nothing

    # Save both figures once (using your function)
    plt.figure(fig_acc.number)
    configure_and_save_plot(
        _plt=plt,
        _x_label="Round",
        _y_label="Accuracy",
        _title="",
        _file_path=f"{out_prefix}_accuracy",
        _legend_handles=acc_handles_all
    )

    plt.figure(fig_loss.number)
    configure_and_save_plot(
        _plt=plt,
        _x_label="Round",
        _y_label="Loss",
        _title="",
        _file_path=f"{out_prefix}_loss",
        _legend_handles=loss_handles_all
    )

### Save plots

In [18]:
from plot_utils.plotting import configure_and_save_plot
import constants

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

Training on cuda using PyTorch 2.8.0+cpu
Training on cuda using PyTorch 2.8.0+cpu
Training on cuda using PyTorch 2.8.0+cpu


1.0. Reading drift specifications from the log

In [19]:
# drift_specs = read_logs('D://publications_related//2026//ICDCS//logs//swap//MNIST//saved_logs_fedavg////drift_specs_log.pkl')  # contains e.g.,
# drift_specs['drift_clustered_client_indices'] =[[[0, 1, 2, 3], [4, 5, 6, 7]],
# [[0, 1], [2, 3, 4, 5, 6, 7]], where [0, 1] -> 1, [2, 3, 4, 5, 6, 7] -> 2
# [[0, 1, 2, 3, 4, 5], [6, 7]]] 
# drift_specs['drift_step_rounds'] = [20, 33, 35, 45]
drift_specs = read_logs('D://publications_related//2026//ICDCS//logs//swap//F_MNIST//saved_logs_fedavg//drift_specs_log.pkl')  # contains e.g.,

drift_clustered_client_indices = drift_specs['drift_clustered_client_indices'] 
# drift_pattern=[[1, 2], [1, 2], [2, 1]]
drift_pattern=[[1, 2], [1, 2]]  # scenario B

reordered_clusters = reorder_clusters_by_pattern(drift_clustered_client_indices, drift_pattern)
print(reordered_clusters)

# This code appends the non drifted client cluster to the reordered_clusters in the previous step

extra_cluster = [8, 9]

reordered_clusters_complete = [
    step_clusters + [extra_cluster]
    for step_clusters in reordered_clusters
]

[[[0], [1, 2, 3, 4, 5, 6, 7]], [[0, 1, 2, 3, 4, 5], [6, 7]]]


2. Configurations handles to what to diplay on the plot and saving plots

In [44]:
#For Sceanio A 
drift_clustered_client_indices[0] = drift_clustered_client_indices[0][0]+drift_clustered_client_indices[0][1]
drift_clustered_client_indices

In [21]:
# Logs
fedavg_log  = read_logs('D://publications_related//2026//ICDCS//logs//swap//F_MNIST//saved_logs_fedavg//client_log.pkl')
# fedex_log   = read_logs('D://publications_related//2026//ICDCS//logs//swap//F_MNIST//saved_logs_fedex//client_log.pkl')
fedex_log   = read_logs('D://publications_related//2026//ICDCS//logs//swap//test//saved_logs_fedex//client_log.pkl')
oracle_log  = read_logs('D://publications_related//2026//ICDCS//logs//swap/F_MNIST///saved_logs_oracle//client_log.pkl')

drift_steps = drift_specs["drift_step_rounds"]  # identical for all
clusters_no_final = reordered_clusters_complete  # identical for all

# Build plot inputs per method
fedavg_pre,  fedavg_post  = build_plot_inputs_for_log(fedavg_log,  drift_steps, clusters_no_final)
fedex_pre,   fedex_post   = build_plot_inputs_for_log(fedex_log,   drift_steps, clusters_no_final)
oracle_pre,  oracle_post  = build_plot_inputs_for_log(oracle_log,  drift_steps, clusters_no_final)

# Shared cluster linestyles (same cluster mapping across methods)
cluster_linestyles = {0: "--", 1: "-.", 2: ":"}

method_configs = [
    {
        "name": "FedAvg",
        "pre_stats": fedavg_pre,
        "post_cluster_plot_stats": fedavg_post,
        "color": "red",
        "cluster_linestyles": cluster_linestyles,
        "show_in_legend": True,
        "legend_mode": "method"        # ONE entry only
    },
    {
        "name": "FedEx",
        "pre_stats": fedex_pre,
        "post_cluster_plot_stats": fedex_post,
        "color": "blue",
        "cluster_linestyles": cluster_linestyles,
        "show_in_legend": True,
        "legend_mode": "method"
    },
    {
        "name": "Oracle",
        "pre_stats": oracle_pre,
        "post_cluster_plot_stats": oracle_post,
        "color": "green",
        "cluster_linestyles": cluster_linestyles,
        "show_in_legend": True       # plotted but hidden from legend
    }
]

# File save directory and file name
file_name = '1_clients_performance_F_MNIST'
out_prefix = constants.Paths.PLOT_SAVE_PATH + file_name
# Ensure directory exists if needed:
# import os; os.makedirs("plots/MNIST", exist_ok=True)

plot_multiple_methods_pre_post_drift(
    method_configs=method_configs,
    drift_steps=drift_steps,
    out_prefix=out_prefix,
    band_alpha=0.25,
    show_band=True,
    stitch=True
)

### Create and save custom legends 

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

def save_legend_only(
    *,
    handles,
    labels,
    file_path_no_ext,   # e.g., ".../legend_only"
    ncol=None,
    fontsize=8,
    handlelength=2.5,
    columnspacing=1.5
):
    """
    Saves a legend as a standalone figure (PNG + PDF).
    """
    if ncol is None:
        ncol = len(handles)

    fig = plt.figure(figsize=(max(6, 0.9 * len(handles)), 0.8))  # wide & short
    fig.legend(
        handles,
        labels,
        loc="center",
        ncol=ncol,
        frameon=False,
        fontsize=fontsize,
        handlelength=handlelength,
        columnspacing=columnspacing,
    )
    plt.axis("off")

    fig.savefig(f"{file_path_no_ext}.png", dpi=300, bbox_inches="tight", pad_inches=0.05)
    fig.savefig(f"{file_path_no_ext}.pdf", format="pdf", bbox_inches="tight", pad_inches=0.05)
    plt.close(fig)

In [48]:
def build_custom_legend_handles():
    # ---- method styles (example; adjust to your actual plot styles) ----
    method_items = [
        ("FedAvg", "red",   "-"),    # (label, color, linestyle)
        ("FedEx",  "blue",  "-"),
        ("Oracle", "green", "-"),
    ]

    # ---- drift/cluster styles in black (example; adjust to your meaning) ----
    drift_items = [
        ("Drift 1",  "black", "--"),
        ("Drift 2",  "black", "-."),
        ("No drift", "black", ":"),   # or "-" if you prefer solid for no-drift
    ]

    handles = []
    labels = []

    for label, color, ls in method_items + drift_items:
        handles.append(Line2D([0], [0], color=color, linestyle=ls, linewidth=1.8))
        labels.append(label)

    return handles, labels

In [ ]:
# Save legend
handles, labels = build_custom_legend_handles()

legend_out = constants.Paths.PLOT_SAVE_PATH + "legend_only"
save_legend_only(
    handles=handles,
    labels=labels,
    file_path_no_ext=legend_out,
    ncol=len(handles)   # all in one row
)

### Plot seperate graphs based on drift types

1. Detailed axes plot information

In [15]:
import numpy as np
import matplotlib.pyplot as plt


def plot_pre_post_single_cluster_on_axis(
    ax,
    *,
    pre_stats,
    post_cluster_plot_stats,
    cluster_idx,
    metric_prefix,              # "acc" or "loss"
    base_color,
    method_label,
    pre_linestyle="-",
    pre_linewidth=1.8,
    post_linestyle="--",        # <-- drift-type controlled (passed in)
    post_linewidth=1.6,
    band_alpha=0.25,
    show_band=True,
    stitch=True,
    drift_steps=None,
    collect_legend=True,
    smooth_window=1,            # optional: keep if you use smoothing elsewhere
    smooth_mode="moving_avg",   # optional
):
    """
    Plot one method on a given axis:
      - pre-drift (all clients) avg + band
      - post-drift ONLY for one cluster_idx avg + band
      - stitching line from last pre point to first post point (for this cluster)
      - optional drift boundary lines

    Returns: legend handles (typically one handle for the method).
    """

    legend_handles = []

    # ---- Pre-drift ----
    x_pre, y_pre_min, y_pre_avg, y_pre_max = stats_dict_to_arrays(pre_stats, metric_prefix)

    # Average line (pre)
    pre_line, = ax.plot(
        x_pre, y_pre_avg,
        color=base_color,
        linestyle=pre_linestyle,
        linewidth=pre_linewidth,
        label=method_label
    )
    if collect_legend:
        legend_handles.append(pre_line)

    # Band (pre)
    if show_band:
        ax.fill_between(
            x_pre, y_pre_min, y_pre_max,
            color=base_color,
            alpha=band_alpha,
            linewidth=0
        )

    pre_last_round = int(x_pre[-1])
    pre_last_avg = float(y_pre_avg[-1])

    # ---- Drift boundary markers ----
    if drift_steps is not None:
        for d in drift_steps:
            ax.axvline(
                x=d,
                color="black",
                linestyle=":",
                linewidth=0.8,
                alpha=0.6,
                zorder=0
            )

    # ---- Post-drift: ONLY selected cluster ----
    if cluster_idx not in post_cluster_plot_stats:
        return legend_handles

    segments_dict = post_cluster_plot_stats[cluster_idx]
    if not segments_dict:
        return legend_handles

    # Concatenate segments in time order (continuous across segments)
    x_all, y_min_all, y_avg_all, y_max_all = [], [], [], []

    for (start, end) in sorted(segments_dict.keys()):
        seg = segments_dict[(start, end)]
        if len(seg["round"]) == 0:
            continue

        x_seg = np.array(seg["round"], dtype=int)
        y_min_seg = np.array(seg[f"{metric_prefix}_min"], dtype=float)
        y_avg_seg = np.array(seg[f"{metric_prefix}_avg"], dtype=float)
        y_max_seg = np.array(seg[f"{metric_prefix}_max"], dtype=float)

        x_all.append(x_seg)
        y_min_all.append(y_min_seg)
        y_avg_all.append(y_avg_seg)
        y_max_all.append(y_max_seg)

    if not x_all:
        return legend_handles

    x_post = np.concatenate(x_all)
    y_post_min = np.concatenate(y_min_all)
    y_post_avg = np.concatenate(y_avg_all)
    y_post_max = np.concatenate(y_max_all)

    # ---- Stitch pre -> post for this cluster ----
    if stitch and len(x_post) > 0:
        first_post_round = int(x_post[0])
        first_post_avg = float(y_post_avg[0])
        if pre_last_round + 1 == first_post_round:
            ax.plot(
                [pre_last_round, first_post_round],
                [pre_last_avg, first_post_avg],
                color=base_color,
                linestyle=pre_linestyle,
                linewidth=pre_linewidth
            )

    # ---- Post line (cluster) ----
    ax.plot(
        x_post, y_post_avg,
        color=base_color,
        linestyle=post_linestyle,   # <-- drift-type linestyle comes from caller
        linewidth=post_linewidth
    )

    # Band (post)
    if show_band:
        ax.fill_between(
            x_post, y_post_min, y_post_max,
            color=base_color,
            alpha=band_alpha,
            linewidth=0
        )

    return legend_handles

2. Config handling and saving plots

In [16]:
def plot_one_cluster_across_methods(
    *,
    method_configs,
    cluster_idx,
    metric_prefix,           # "acc" or "loss"
    drift_steps,
    out_file_path,           # full prefix (no extension)
    post_linestyle="--",     # <-- drift-type linestyle controlled from execution block
    title="",
    band_alpha=0.25,
    show_band=True,
    stitch=True,
    smooth_window=1,         # optional
    smooth_mode="moving_avg" # optional
):
    """
    Create one figure for ONE cluster (cluster_idx) and ONE metric (acc/loss),
    overlaying all methods (FedAvg/FedEx/Oracle) with different colors.
    """

    fig, ax = plt.subplots()

    drift_steps_sorted = sorted(drift_steps)
    legend_handles = []

    for cfg in method_configs:
        handles = plot_pre_post_single_cluster_on_axis(
            ax,
            pre_stats=cfg["pre_stats"],
            post_cluster_plot_stats=cfg["post_cluster_plot_stats"],
            cluster_idx=cluster_idx,
            metric_prefix=metric_prefix,
            base_color=cfg.get("color", "black"),
            method_label=cfg.get("name", "method"),
            pre_linestyle="-",
            pre_linewidth=1.8,
            post_linestyle=post_linestyle,   # <-- same linestyle for all methods in this drift-type plot
            post_linewidth=1.6,
            band_alpha=band_alpha,
            show_band=show_band,
            stitch=stitch,
            drift_steps=drift_steps_sorted,
            collect_legend=True,
            smooth_window=smooth_window,
            smooth_mode=smooth_mode
        )
        legend_handles.extend(handles)

    plt.figure(fig.number)
    configure_and_save_plot(
        _plt=plt,
        _x_label="Round",
        _y_label="Accuracy" if metric_prefix == "acc" else "Loss",
        _title=title,
        _file_path=out_file_path,
        _legend_handles=None
    )


3. Plotting function

In [17]:
def plot_all_clusters_separate(
    *,
    method_configs,
    drift_steps,
    cluster_map,              # e.g., {0:"drift_1", 1:"drift_2", 2:"no_drift"}
    drift_type_linestyles,    # e.g., {"drift_1":"--", "drift_2":"-.", "no_drift":":"}
    out_prefix,
    band_alpha=0.25,
    show_band=True,
    stitch=True,
    smooth_window=1,
    smooth_mode="moving_avg",
):
    """
    Generates separate figures per drift type / cluster:
      - Accuracy for each drift type
      - Loss for each drift type

    Drift-type linestyles are controlled from the execution block via drift_type_linestyles.
    """

    for cluster_idx, drift_name in cluster_map.items():
        post_ls = drift_type_linestyles.get(drift_name, "--")  # fallback

        # Accuracy plot for this drift type
        plot_one_cluster_across_methods(
            method_configs=method_configs,
            cluster_idx=cluster_idx,
            metric_prefix="acc",
            drift_steps=drift_steps,
            out_file_path=f"{out_prefix}_{drift_name}_accuracy",
            post_linestyle=post_ls,
            title=f"Accuracy (Drift type: {drift_name})",
            band_alpha=band_alpha,
            show_band=show_band,
            stitch=stitch,
            smooth_window=smooth_window,
            smooth_mode=smooth_mode
        )

        # Loss plot for this drift type
        plot_one_cluster_across_methods(
            method_configs=method_configs,
            cluster_idx=cluster_idx,
            metric_prefix="loss",
            drift_steps=drift_steps,
            out_file_path=f"{out_prefix}_{drift_name}_loss",
            post_linestyle=post_ls,
            title=f"Loss (Drift type: {drift_name})",
            band_alpha=band_alpha,
            show_band=show_band,
            stitch=stitch,
            smooth_window=smooth_window,
            smooth_mode=smooth_mode
        )

4. Execute

In [18]:
cluster_map = {0: "drift_1", 1: "drift_2", 2: "no_drift"}

drift_type_linestyles = {
    "drift_1": "--",
    "drift_2": "-.",
    "no_drift": ":"
}

file_name = "3_clusterwise_comparison_F_MNIST"
out_prefix = constants.Paths.PLOT_SAVE_PATH + file_name

plot_all_clusters_separate(
    method_configs=method_configs,   # FedAvg/FedEx/Oracle
    drift_steps=drift_steps,
    cluster_map=cluster_map,
    drift_type_linestyles=drift_type_linestyles,
    out_prefix=out_prefix,
    band_alpha=0.25,
    show_band=True,
    stitch=True,
    smooth_window=9,                 # optional smoothing
    smooth_mode="moving_avg"
)